# 911 Calls

In this chapter we will be using data on 911 calls in Fernhaven to estimate the probability of having no 911 calls in a minute. This distribution is modeled using a Poisson distribution with probability mass function 

$$
    p(k)=\frac{\mu^k}{k!}e^{-\mu}
$$ 
where $\mu$ is the expected number of calls in one minute.

We will be covering two methods of estimation: proportion and method of moments/maximum likelihood estimate (MLE).

The first step is to perform *data wrangling* on the dataset to get several datasets of the number of calls per minute with $n=60$ (i.e. an hour long). In order to assume the same distribution across these datasets we will use the same time of day for each dataset.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math

In [ ]:
# Read in the data and extract calls between 6 and 7
df = pd.read_csv("data/911_Calls.csv")

df["datetime"] = pd.to_datetime(df["Date"] + " " + df["Call time"])

df["date"] = df["datetime"].dt.date.astype(str)
df["hour"] = df["datetime"].dt.hour
df["minute"] = df["datetime"].dt.minute

df_hour = df[df["hour"] == 6]
counts = df_hour.groupby(["date", "minute"]).size().reset_index(name="count")

hourly_counts = {}
for date, grp in counts.groupby("date"):
    series = grp.set_index("minute")["count"]
    full = series.reindex(range(60), fill_value=0)
    hourly_counts[date] = full.tolist()

Now that we have our datasets we can compute our estimates. 

The first estimate we will compute is the proportion of minutes with no calls in each dataset ($\hat{p}$). 

$$
    \hat{p}=\frac{\text{number of minutes with 0 calls}}{60}
$$


In [ ]:
p_hat = [counts.count(0) / 60.0 for counts in hourly_counts.values()]

p_hat

The second estimate is the MLE ($e^{-\hat{\mu}}$). To do so, we need to calculate an estimate for $\mu$. For each dataset we calculate $\hat{\mu}$ with the formula: 

$$
    \hat{\mu}=\frac{\text{number of calls}}{60}=\frac{\text{sum of calls per minute}}{60}
$$

Once we have calculated an estimate for $\mu$, we can use the probability mass function of the Poisson distribution where $k=0$ to calculate $e^{-\hat{\mu}}$. 
<!-- 
$$
    \hat{p}=\frac{\mu^0}{0!}e^{-\mu}
$$ 

$$
    \hat{p}=e^{-\mu}
$$ -->

In [ ]:
mle = [math.exp(-1 * (sum(counts) / 60.0)) for counts in hourly_counts.values()]

mle

We can now make histograms of the obtained estimates for both cases to estimate the distribution and indicate the deviation from the true parameter. The true parameter in this case will be obtained using the full data-set.

In [ ]:
# compute the true parameter over the full dataset
call_counts_full_dataset = (
    df.groupby(["date", "hour", "minute"]).size().reset_index(name="count")
)
full_counts = []
for (_, _), grp in call_counts_full_dataset.groupby(["date", "hour"]):
    ser = grp.set_index("minute")["count"]
    full_counts.extend(ser.reindex(range(60), fill_value=0).tolist())

true_parameter = full_counts.count(0) / len(full_counts)
print(f"Estimated p (full dataset) = {true_parameter}")

# plot estimates
plt.figure(figsize=(15, 6))
plt.subplot(1, 2, 1)
plt.hist(p_hat)
plt.title("Estimation $\hat{p}$")
plt.ylabel("frequency")

plt.axvline(x=true_parameter, color="r", linestyle="--")

plt.subplot(1, 2, 2)
plt.hist(mle)
plt.title("Estimation $e^{-\mu}$")
plt.ylabel("frequency")

plt.axvline(x=true_parameter, color="r", linestyle="--")

plt.show()

Given these two estimators, we need to determine the best one. $\hat{p}$ is an unbiased estimator, but $e^{-\hat{\mu}}$ is positively biased. From the histograms, we also observe that $\hat{p}$ has a larger variance than $e^{-\hat{\mu}}$. This translates to being typically far away from the true value versus being typically close to a value above the true value. We typically want to select the estimator with the lowest mean squared error (MSE).

Using the true value from the dataset we can estimate the MSE value for both estimators. Because $\hat{p}$ is unbiased, the MSE for $\hat{p}$ will be equal to the variance, which is $\frac{p(1-p)}{n}$. The second estimator, $e^{-\hat{\mu}}$, is positively biased and can be written as a function of $p$ to be $(p^{n(1-e^{-\frac{1}{n}})}-p)^2 + (p^{n(1-e^{-\frac{2}{n}})}-p^{2n(1-e^{-\frac{1}{n}})})$. Given $p=0.2108$ and $n=60$ in this scenario, the MSE is about $0.0028$ for $\hat{p}$ and $0.0012$ for $e^{-\hat{\mu}}$. 